In [1]:
print('importing modules')
import os
import sys
import json
import argparse
import numpy as np
import math
from einops import rearrange
import time
import random
import string
import h5py
from tqdm import tqdm
import webdataset as wds

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from accelerate import Accelerator, DeepSpeedPlugin

from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder
from models import GNet8_Encoder

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

# custom functions #
import utils

### Multi-GPU config ###
local_rank = os.getenv('RANK')
if local_rank is None: 
    local_rank = 0
else:
    local_rank = int(local_rank)
print("LOCAL RANK ", local_rank)  

accelerator = Accelerator(split_batches=False, mixed_precision="fp16") # ['no', 'fp8', 'fp16', 'bf16']

print("PID of this process =",os.getpid())
device = accelerator.device
print("device:",device)
world_size = accelerator.state.num_processes
distributed = not accelerator.state.distributed_type == 'NO'
num_devices = torch.cuda.device_count()
if num_devices==0 or not distributed: num_devices = 1
num_workers = num_devices
print(accelerator.state)

print("distributed =",distributed, "num_devices =", num_devices, "local rank =", local_rank, "world size =", world_size)
print = accelerator.print # only print if local_rank=0

importing modules


/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


LOCAL RANK  0
PID of this process = 1162506
device: cuda
Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

distributed = False num_devices = 1 local rank = 0 world size = 1


In [2]:
# Load embedding model (last hidden layer)
try:
    print(clip_img_embedder)
except:
    clip_img_embedder = FrozenOpenCLIPImageEmbedder(
        arch="ViT-bigG-14",
        version="laion2b_s39b_b160k",
        output_tokens=True,
        only_tokens=True,
    )
    clip_img_embedder.to(device)
clip_seq_dim = 256
clip_emb_dim = 1664

## Load embedding model (last layer)
#     clip_img_embedder = FrozenOpenCLIPImageEmbedder(
#         arch="ViT-bigG-14",
#         version="laion2b_s39b_b160k",
#         output_tokens=False,
#         only_tokens=False,
#     )
#     clip_img_embedder.to(device)
# clip_seq_dim = 1
# clip_emb_dim = 1280

In [3]:
plot_all = False
compute_circular = False  # for the circular tests looking at image similarity in clip space (without any brain data involved)
saving = True

# if running this interactively, can specify jupyter_args here for argparser to use
if utils.is_interactive():
    model_name = "sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15"
    eval_dir = f"/scratch/am10150/projects/MindEyeV2/src/mindeyev2/evals/{model_name}"
    if ("remove" in model_name and "random" in model_name) or "ses-04" in model_name:
        all_recons_path = f"{eval_dir}/all_recons.pt"
    elif "paul" in model_name:
        all_recons_path = f"evals/{model_name}/{model_name}_all_recons.pt"
    else:
        all_recons_path = f"{eval_dir}/{model_name}_all_recons.pt" 

    data_path = "/scratch/am10150/projects/MindEyeV2/src/mindeyev2"
    print("model_name:", model_name)

    jupyter_args = f"--model_name={model_name} --data_path={data_path} --all_recons_path={all_recons_path} --eval_dir={eval_dir}"
    print(jupyter_args)
    jupyter_args = jupyter_args.split()
    
    from IPython.display import clear_output # function to clear print outputs in cell
    %load_ext autoreload 
    # this allows you to change functions in models.py or utils.py and have this notebook automatically update with your revisions
    %autoreload 2 

model_name: sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15
--model_name=sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15 --data_path=/scratch/am10150/projects/MindEyeV2/src/mindeyev2 --all_recons_path=/scratch/am10150/projects/MindEyeV2/src/mindeyev2/evals/sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15/sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15_all_recons.pt --eval_dir=/scratch/am10150/projects/MindEyeV2/src/mindeyev2/evals/sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15


In [4]:
parser = argparse.ArgumentParser(description="Model Training Configuration")
parser.add_argument(
    "--model_name", type=str, default="testing",
    help="name of model, used for ckpt saving and wandb logging (if enabled)",
)
parser.add_argument(
    "--data_path", type=str, default="/weka/proj-fmri/shared/mindeyev2_dataset",
    help="Path to where NSD data is stored / where to download it to",
)
parser.add_argument(
    "--all_recons_path", type=str,
    help="Path to where all_recons.pt is stored",
)

parser.add_argument(
    "--eval_dir", type=str,
    help="Path to where evaluations should be stored",
)

parser.add_argument(
    "--seed",type=int,default=42,
)
if utils.is_interactive():
    args = parser.parse_args(jupyter_args)
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)
    
# seed all random functions
utils.seed_everything(seed)

# Evals

In [5]:
if ("remove" in model_name and "random" in model_name) or "ses-04" in model_name:
    all_images = torch.load(f"{eval_dir}/all_images.pt")
    all_clipvoxels = torch.load(f"{eval_dir}/all_clipvoxels.pt")
    all_predcaptions = torch.load(f"{eval_dir}/all_predcaptions.pt")
    all_unrefinedrecons = torch.load(f"{eval_dir}/all_recons.pt")
elif "ses-01" in model_name and "paul" in model_name:
    all_images = torch.load(f"evals/{model_name}/{model_name}_all_images.pt")
    all_clipvoxels = torch.load(f"evals/{model_name}/{model_name}_all_clipvoxels.pt")
    all_predcaptions = torch.load(f"evals/{model_name}/{model_name}_all_predcaptions.pt")
    all_unrefinedrecons = torch.load(f"evals/{model_name}/{model_name}_all_recons.pt")
else:
    all_images = torch.load(f"{eval_dir}/{model_name}_all_images.pt") 
    all_clipvoxels = torch.load(f"{eval_dir}/{model_name}_all_clipvoxels.pt") 
    all_predcaptions = torch.load(f"{eval_dir}/{model_name}_all_predcaptions.pt") 
    all_unrefinedrecons = torch.load(f"{eval_dir}/{model_name}_all_recons.pt") 

print(all_images.shape)
print("all_recons_path:", all_recons_path)
all_recons = torch.load(all_recons_path)

# all_blurryrecons = torch.load(f"{eval_dir}/all_blurryrecons.pt")

torch.Size([62, 3, 256, 256])
all_recons_path: /scratch/am10150/projects/MindEyeV2/src/mindeyev2/evals/sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15/sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15_all_recons.pt


In [6]:
# if "ses-01" in model_name:
#     paul_all_images = torch.load(f"evals/sub-001_ses-01_bs24_MST_paul_MSTsplit/sub-001_ses-01_bs24_MST_paul_MSTsplit_all_images.pt").to('cpu')
#     paul_all_clipvoxels = torch.load(f"evals/sub-001_ses-01_bs24_MST_paul_MSTsplit/sub-001_ses-01_bs24_MST_paul_MSTsplit_all_clipvoxels.pt").to('cpu')
#     paul_all_recons = torch.load(f"evals/sub-001_ses-01_bs24_MST_paul_MSTsplit/sub-001_ses-01_bs24_MST_paul_MSTsplit_all_recons.pt").to('cpu')
#     # paul_all_prior_out = torch.load(f"evals/sub-001_ses-01_bs24_MST_paul_MSTsplit/sub-001_ses-01_bs24_MST_paul_MSTsplit_all_prior_out.pt").to('cpu')
#     # all_images = torch.load(f"{eval_dir}/all_images.pt") 
#     print(paul_all_images.shape, all_images.shape)
#     print(paul_all_clipvoxels.shape, all_clipvoxels.shape)
#     print(torch.eq(paul_all_clipvoxels, all_clipvoxels))
#     # assert torch.allclose(paul_all_images, all_images)

In [7]:
# for i in range(100):
#     # print(torch.allclose(paul_all_images[i], all_images[i]))
#     pass

In [8]:
# num_images = paul_all_images.size(0)
# rows = 10  # Number of rows for the grid
# cols = 10  # Number of columns for the grid

# fig, axes = plt.subplots(rows, cols * 2, figsize=(80, 40))

# for i in range(num_images):
#     row = i // cols
#     col = (i % cols) * 2  # Adjust for side-by-side
    
#     # Plot correct image
#     ax_correct = axes[row, col]
#     ax_correct.imshow(paul_all_recons[i].permute(1, 2, 0).cpu().numpy())
#     ax_correct.axis('off')
#     ax_correct.set_title(f"Correct {i}")
    
#     # Plot modified image
#     ax_modified = axes[row, col + 1]
#     ax_modified.imshow(all_recons[i].permute(1, 2, 0).cpu().numpy())
#     ax_modified.axis('off')
#     ax_modified.set_title(f"Modified {i}")

# plt.tight_layout()
# plt.show()

In [9]:
model_name_plus_suffix = all_recons_path.split('/')[-1]
print(model_name_plus_suffix)
print(all_images.shape, all_recons.shape)

sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15_all_recons.pt
torch.Size([62, 3, 256, 256]) torch.Size([62, 3, 256, 256])


In [6]:
# if "MST" in model_name:
    # if ("remove" in model_name and "random" in model_name) or "ses-04" in model_name or "rishab" in model_name:
    #     MST_ID = np.load(f"{eval_dir}/MST_ID.npy")
    #     MST_pairmate_indices = np.load(f"{eval_dir}/MST_pairmate_indices.npy")
    # elif "paul" in model_name:
    #     MST_ID = np.load(f"evals/{model_name}/{model_name}_MST_ID.npy")
    #     MST_pairmate_indices = np.array(utils.find_paired_indices(torch.Tensor(MST_ID)))
        # print(MST_pairmate_indices)
    # else:
    #     MST_ID = np.load(f"{eval_dir}/{model_name}_MST_ID.npy") 
    #     MST_pairmate_indices = np.load(f"{eval_dir}/{model_name}_MST_pairmate_indices.npy") 

    # pairs = utils.find_paired_indices(torch.Tensor(MST_ID))
    # if "close_to_MST" in model_name or ("remove" in model_name and "random" in model_name) or "ses-0" in model_name:
    #     pairs = np.array(pairs[:-1])  # index out the placeholder
    # pairs = np.array(pairs)
    # if "ses-0" in model_name:
    #     if "ses-01" in model_name or "ses-04" in model_name:
    #         print(pairs.shape)
    #         assert pairs.shape == (49,2)
    #     else:
    #         assert pairs.shape == (50,3)
    # else:
    #     assert pairs.shape == (100,3)
    # print(pairs)
    # repeats_in_test = torch.load(f"{eval_dir}/repeats_in_test.pt")
    # test_image_indices = torch.load(f"{eval_dir}/test_image_indices.pt")
    # all_unique_images = all_images[MST_pairmate_indices.flatten()]
    # all_unique_clipvoxels = all_clipvoxels[MST_pairmate_indices.flatten()]
all_unique_images = all_images
all_unique_clipvoxels = all_clipvoxels
print(model_name, all_unique_images.shape, all_unique_clipvoxels.shape)

# print(model_name, MST_pairmate_indices.shape, all_unique_images.shape, all_unique_clipvoxels.shape)

sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_1_avgrepeats_finalmask_epochs_15 (31, 2) torch.Size([62, 3, 256, 256]) torch.Size([62, 256, 1664])


In [7]:
# visualize all unique images
if plot_all:
    # Plot all the MST images and pairmates
    import textwrap
    def wrap_title(title, wrap_width):
        return "\n".join(textwrap.wrap(title, wrap_width))

    size = int(np.ceil(MST_pairmate_indices.shape[0]/2))  # helps determine size of plot
    fig, axes = plt.subplots(size, 4, figsize=(15, size*4))
    jj=-1; kk=0;
    for i, j in enumerate(all_unique_images):
        jj+=1
        axes[kk][jj].imshow(utils.torch_to_Image(j))
        axes[kk][jj].axis('off')
        if jj==3: 
            kk+=1; jj=-1

    fig.tight_layout()
    # plt.savefig('figures/MST_2_pairmates_10-01')
    plt.show()

In [10]:
imsize = 256
if all_images.shape[-1] != imsize:
    all_images = transforms.Resize((imsize,imsize))(all_images).float()
if all_recons.shape[-1] != imsize:
    all_recons = transforms.Resize((imsize,imsize))(all_recons).float()
try:
    if all_blurryrecons.shape[-1] != imsize:
        all_blurryrecons = transforms.Resize((imsize,imsize))(all_blurryrecons).float()
except:
    pass

if "enhanced" in model_name_plus_suffix:
    try:
        all_recons = all_recons*.75 + all_blurryrecons*.25
        print("weighted averaging to improve low-level evals")
    except:
        pass

In [11]:
# visualize some images with recons and captions
plot_all = False
if plot_all:
    assert np.all(all_images.shape == all_recons.shape)
    import textwrap
    def wrap_title(title, wrap_width):
        return "\n".join(textwrap.wrap(title, wrap_width))

    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    jj=-1; kk=0;
    for j in np.array([0,1,2,3,4,5]):
        jj+=1
        # print(kk,jj)
        axes[kk][jj].imshow(utils.torch_to_Image(all_images[j]))
        axes[kk][jj].axis('off')
        jj+=1
        axes[kk][jj].imshow(utils.torch_to_Image(all_recons[j]))
        axes[kk][jj].axis('off')
        axes[kk][jj].set_title(wrap_title(str(all_predcaptions[[j]]),wrap_width=30), fontsize=8)
        if jj==3: 
            kk+=1; jj=-1

    fig.tight_layout()
    # plt.savefig('figures/recon_09-26')
    plt.show()

# Retrieval eval (chance =  1/100)

In [12]:
from scipy import stats

all_fwd_acc = []
all_bwd_acc = []

assert len(all_unique_images) == len(all_unique_clipvoxels)  

all_percent_correct_fwds, all_percent_correct_bwds = [], []

with torch.cuda.amp.autocast(dtype=torch.float16):
    all_emb = clip_img_embedder(all_unique_images.to(device)).float() # CLIP-Image
    all_emb_ = all_unique_clipvoxels # CLIP-Brain

    # flatten if necessary
    all_emb = all_emb.reshape(len(all_emb),-1).to(device)
    all_emb_ = all_emb_.reshape(len(all_emb_),-1).to(device)

    # l2norm 
    all_emb = nn.functional.normalize(all_emb,dim=-1)
    all_emb_ = nn.functional.normalize(all_emb_,dim=-1)

    all_labels = torch.arange(len(all_emb)).to(device)
    all_bwd_sim = utils.batchwise_cosine_similarity(all_emb,all_emb_)  # clip, brain
    all_fwd_sim = utils.batchwise_cosine_similarity(all_emb_,all_emb)  # brain, clip

    # if "ses-0" not in model_name or "ses-01" in model_name or "ses-04" in model_name:
    #     assert len(all_fwd_sim) == 100
    #     assert len(all_bwd_sim) == 100
    # else:
    #     assert len(all_fwd_sim) == 50
    #     assert len(all_bwd_sim) == 50
    
    all_percent_correct_fwds = utils.topk(all_fwd_sim, all_labels, k=1).item()
    all_percent_correct_bwds = utils.topk(all_bwd_sim, all_labels, k=1).item()

all_fwd_acc.append(all_percent_correct_fwds)
all_bwd_acc.append(all_percent_correct_bwds)

all_fwd_sim = np.array(all_fwd_sim.cpu())
all_bwd_sim = np.array(all_bwd_sim.cpu())

print(f"overall fwd percent_correct: {all_fwd_acc[0]:.4f}")
print(f"overall bwd percent_correct: {all_bwd_acc[0]:.4f}")

overall fwd percent_correct: 0.4677
overall bwd percent_correct: 0.4032


In [16]:
# top-n predictions using CLIP brain embeddings

if plot_all:
    use_fwd_sim = True
    top_n = 10  # how many of the top n images to display
    print("Given Brain embedding, find correct Image embedding")
    fig, ax = plt.subplots(nrows=len(all_unique_images), ncols=top_n+1, figsize=(top_n*2,len(all_unique_images)*2))
    for trial in range(len(all_unique_images)):
        ax[trial, 0].imshow(utils.torch_to_Image(all_unique_images[trial]))
        ax[trial, 0].set_title("original\nimage")
        ax[trial, 0].axis("off")
        for attempt in range(top_n):
            if trial < 50:
                if "ses-0" not in model_name:
                    sim_half = fwd_sim_halves[0] if use_fwd_sim else bwd_sim_halves[0]
                    unique_imgs_to_plot = all_unique_images[:int(len(all_unique_images)/2)]
                    # unique_clipvoxels_to_plot = all_unique_clipvoxels[:int(len(all_unique_clipvoxels)/2)]
                else:
                    sim_half = all_fwd_sim if use_fwd_sim else all_bwd_sim
                    unique_imgs_to_plot = all_unique_images
                    # unique_clipvoxels_to_plot = all_unique_clipvoxels
                which = np.flip(np.argsort(sim_half[trial]))[attempt]

            elif trial >= 50:
                if "ses-0" not in model_name:
                    sim_halves = fwd_sim_halves[1] if use_fwd_sim else bwd_sim_halves[1]
                    unique_imgs_to_plot = all_unique_images[int(len(all_unique_images)/2):]
                    # unique_clipvoxels_to_plot = all_unique_clipvoxels[int(len(all_unique_clipvoxels)/2):]
                else:
                    sim_halves = all_fwd_sim if use_fwd_sim else all_bwd_sim
                    unique_imgs_to_plot = all_unique_images
                    # unique_clipvoxels_to_plot = all_unique_clipvoxels
                which = np.flip(np.argsort(sim_half[trial-50]))[attempt]

            ax[trial, attempt+1].imshow(utils.torch_to_Image(unique_imgs_to_plot[which]))
            ax[trial, attempt+1].set_title(f"Top {attempt+1}")
            ax[trial, attempt+1].axis("off")
    fig.tight_layout()
    # plt.savefig('figures/retrieval_top10')
    plt.show()

## 2-way identification

In [24]:
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

@torch.no_grad()
def two_way_identification(all_recons, all_images, model, preprocess, feature_layer=None, return_avg=True):
    preds = model(torch.stack([preprocess(recon) for recon in all_recons], dim=0).to(device))
    reals = model(torch.stack([preprocess(indiv) for indiv in all_images], dim=0).to(device))
    if feature_layer is None:
        preds = preds.float().flatten(1).cpu().numpy()
        reals = reals.float().flatten(1).cpu().numpy()
    else:
        preds = preds[feature_layer].float().flatten(1).cpu().numpy()
        reals = reals[feature_layer].float().flatten(1).cpu().numpy()

    r = np.corrcoef(reals, preds)
    r = r[:len(all_images), len(all_images):]
    congruents = np.diag(r)

    success = r < congruents
    success_cnt = np.sum(success, 0)

    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images)-1)
        return perf
    else:
        return success_cnt, len(all_images)-1
    
all_recons=all_recons.to(device)
all_images=all_images.to(device)

## PixCorr

In [25]:
preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
])

# Flatten images while keeping the batch dimension
all_images_flattened = preprocess(all_images).reshape(len(all_images), -1).cpu()
all_recons_flattened = preprocess(all_recons).view(len(all_recons), -1).cpu()

print(all_images_flattened.shape)
print(all_recons_flattened.shape)

corr_stack = []

corrsum = 0
for i in tqdm(range(len(all_images))):
    corrcoef = np.corrcoef(all_images_flattened[i], all_recons_flattened[i])[0][1]
    if np.isnan(corrcoef):
        print("WARNING: CORRCOEF WAS NAN")
        corrcoef = 0
    corrsum += corrcoef
    corr_stack.append(corrcoef)
corrmean = corrsum / len(all_images)

pixcorr = corrmean
print(pixcorr)

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


torch.Size([62, 541875])
torch.Size([62, 541875])


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:00<00:00, 186.13it/s]

0.18603376155715132


In [26]:
# print(all_images.shape)
# print(all_images_flattened.shape)
# print(all_recons.shape)
# print(all_recons_flattened.shape)
# len(all_images)

## SSIM

In [27]:
# see https://github.com/zijin-gu/meshconv-decoding/issues/3
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim

preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR), 
])

# convert image to grayscale with rgb2grey
img_gray = rgb2gray(preprocess(all_images).permute((0,2,3,1)).cpu())
recon_gray = rgb2gray(preprocess(all_recons).permute((0,2,3,1)).cpu())
print("converted, now calculating ssim...")

ssim_score=[]
for im,rec in tqdm(zip(img_gray,recon_gray),total=len(all_images)):
    ssim_score.append(ssim(rec, im, multichannel=True, gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=1.0))

ssim = np.mean(ssim_score)
print(ssim)

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


converted, now calculating ssim...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:00<00:00, 62.37it/s]

0.28891106856068033


## AlexNet

In [28]:
from torchvision.models import alexnet, AlexNet_Weights
alex_weights = AlexNet_Weights.IMAGENET1K_V1

alex_model = create_feature_extractor(alexnet(weights=alex_weights), return_nodes=['features.4','features.11']).to(device)
alex_model.eval().requires_grad_(False).to(device)

# see alex_weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

layer = 'early, AlexNet(2)'
print(f"\n---{layer}---")
all_per_correct = two_way_identification(all_recons, all_images, 
                                                          alex_model, preprocess, 'features.4')
alexnet2 = np.mean(all_per_correct)
print(f"2-way Percent Correct: {alexnet2:.4f}")

layer = 'mid, AlexNet(5)'
print(f"\n---{layer}---")
all_per_correct = two_way_identification(all_recons, all_images, 
                                                          alex_model, preprocess, 'features.11')
alexnet5 = np.mean(all_per_correct)
print(f"2-way Percent Correct: {alexnet5:.4f}")

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,



---early, AlexNet(2)---
2-way Percent Correct: 0.6631

---mid, AlexNet(5)---
2-way Percent Correct: 0.7232


## InceptionV3

In [29]:
from torchvision.models import inception_v3, Inception_V3_Weights
weights = Inception_V3_Weights.DEFAULT
inception_model = create_feature_extractor(inception_v3(weights=weights), 
                                           return_nodes=['avgpool']).to(device)
inception_model.eval().requires_grad_(False).to(device)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

all_per_correct = two_way_identification(all_recons, all_images,
                                        inception_model, preprocess, 'avgpool')
        
inception = np.mean(all_per_correct)
print(f"2-way Percent Correct: {inception:.4f}")

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/feature_extraction.py:174: UserWarning: NOTE: The nodes obtained by tracing the model in eval mode are a subsequence of those obtained in train mode. When choosing nodes for feature extraction, you may need to specify output nodes for train and eval mode separately.
  warnings.warn(msg + suggestion_msg)


2-way Percent Correct: 0.5582


## CLIP

In [30]:
import clip
clip_model, preprocess = clip.load("ViT-L/14", device=device)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

all_per_correct = two_way_identification(all_recons, all_images,
                                        clip_model.encode_image, preprocess, None) # final layer
clip_ = np.mean(all_per_correct)
print(f"2-way Percent Correct: {clip_:.4f}")

2-way Percent Correct: 0.5325


## Efficient Net

In [31]:
import scipy as sp
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
weights = EfficientNet_B1_Weights.DEFAULT
eff_model = create_feature_extractor(efficientnet_b1(weights=weights), 
                                    return_nodes=['avgpool'])
eff_model.eval().requires_grad_(False).to(device)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

gt = eff_model(preprocess(all_images))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = eff_model(preprocess(all_recons))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

effnet_nomean = np.array([sp.spatial.distance.correlation(gt[i],fake[i]) for i in range(len(gt))])
effnet = effnet_nomean.mean()
print("Distance:",effnet)

Distance: 0.9263052877025891


## SwAV

In [32]:
swav_model = torch.hub.load('facebookresearch/swav:main', 'resnet50')
swav_model = create_feature_extractor(swav_model, 
                                    return_nodes=['avgpool'])
swav_model.eval().requires_grad_(False).to(device)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

gt = swav_model(preprocess(all_images))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = swav_model(preprocess(all_recons))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

swav_nomean = np.array([sp.spatial.distance.correlation(gt[i],fake[i]) for i in range(len(gt))])
swav = swav_nomean.mean()
print("Distance:",swav)

Using cache found in /home/am10150/.cache/torch/hub/facebookresearch_swav_main
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Distance: 0.5789671876351692


In [ ]:
#[pixcorr, ssim, alexnet2, alexnet5, inception, clip_, effnet, swav, percent_correct_fwd, percent_correct_bwd]
import pandas as pd
# pd.options.display.float_format = '{:.2%}'.format
# pd.reset_option('all')
metric_names = [
    "pixcorr",
    "ssim",
    "alexnet2",
    "alexnet5",
    "clip_",
    "inception",
    "effnet",
    "swav"
]
metrics = [
    pixcorr,
    ssim,
    alexnet2,
    alexnet5,
    clip_,
    inception,
    effnet,
    swav,
]
if "ses-0" not in model_name:
    metric_names.extend(["fwd_acc", "bwd_acc", "mst_score"])
    metrics.extend([all_fwd_acc, all_bwd_acc, mst_score])
else:
    metric_names.extend(["fwd_acc", "bwd_acc"])
    metrics.extend([all_fwd_acc[0], all_bwd_acc[0]])

df = pd.DataFrame({'metric': metric_names, 'values': metrics})
print(df)
# print(model_name_plus_suffix)
final_evals_path = f"{eval_dir}/final_evals.csv"
if saving:
    df.to_csv(final_evals_path, index=False)

     metric    values
0   pixcorr  0.186034
1      ssim  0.288911
2  alexnet2  0.663141
3  alexnet5  0.723162
4     clip_  0.532522
5    effnet  0.926305
6      swav  0.578967
7   fwd_acc  0.467742
8   bwd_acc  0.403226


In [34]:
df = pd.read_csv(final_evals_path)
df

,metric,values
0,pixcorr,0.186034
1,ssim,0.288911
2,alexnet2,0.663141
3,alexnet5,0.723162
4,clip_,0.532522
5,effnet,0.926305
6,swav,0.578967
7,fwd_acc,0.467742
8,bwd_acc,0.403226
